# 📖 Notebook 6: Advanced Caching Patterns

You've learned the core patterns: cache-aside, write-through/behind, invalidation, TTL, stampede prevention, and hot keys. Now we'll add **four production-grade techniques** that come up constantly in real systems and interviews.

## Learning Objectives

- Use **negative caching** to stop attackers/bots from hammering your DB with non-existent IDs
- Implement **soft TTL vs hard TTL** (also known as *stale-while-revalidate*)
- Build a **read-through-style wrapper** that hides cache-aside behind a clean API
- Run a **cache warming** script that pre-populates Redis at startup
- Track basic **cache observability metrics** that make all of this measurable

> 💡 This notebook builds on the TTL concepts from Notebook 4 and the wrapper style introduced in Notebook 5.


## 🛠️ Setup

Make sure the infrastructure is running:

```bash
docker compose up -d
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".


In [ ]:
import psycopg2
import redis
import json
import time
import random
from collections import defaultdict

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "caching_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

r = redis.Redis(**REDIS_CONFIG)
r.flushdb()
print("✅ Connected and Redis cleared")

## 📊 A Tiny Metrics Helper

Before building anything, let's add a small helper to count cache events. Without metrics, advanced patterns feel like magic — you can't tell *why* something is fast or stale.

We'll track: hits, misses, **negative hits** (cache says "doesn't exist"), **stale serves** (returned stale data while refreshing), and **refreshes**.

In [ ]:
class CacheMetrics:
    """Tiny helper to count what the cache is doing."""

    def __init__(self):
        self.counts = defaultdict(int)

    def record(self, event: str):
        self.counts[event] += 1

    def report(self):
        total = self.counts["hit"] + self.counts["miss"] + self.counts["negative_hit"]
        hit_rate = ((self.counts["hit"] + self.counts["negative_hit"]) / total * 100) if total else 0
        print("📊 Cache Metrics")
        print("-" * 30)
        for k in ["hit", "miss", "negative_hit", "stale_serve", "refresh", "db_query"]:
            print(f"   {k:<14} {self.counts[k]:>5}")
        print(f"   {'hit_rate':<14} {hit_rate:>5.1f}%")

m = CacheMetrics()
m.record("hit"); m.record("hit"); m.record("miss")
m.report()
print()
print("💡 Every section below will plug into this helper so you can see what's happening.")

## 🚫 Pattern 1: Negative Caching

**Problem:** What happens when someone (or a bot) repeatedly asks for `product:99999999` — a product that doesn't exist?

With plain cache-aside, every request is a cache MISS, every miss hits the database, and the database does work to confirm "still nothing here." This is called **cache penetration** and it's an easy way to overload your DB.

**Fix:** When the DB confirms an item doesn't exist, cache that *fact* with a **sentinel value** (e.g. the string `"__NULL__"`) and a **short TTL**.

### Important rules
1. Use a **sentinel**, not a real `null` — so you can distinguish "not cached" from "cached as missing."
2. Use a **shorter TTL** than for real data — if the item is created later, you don't want to serve "missing" forever.
3. **Never** cache transient errors (timeouts, connection failures) as "missing" — those aren't `404`s, they're outages.

In [ ]:
NEGATIVE_TTL = 30   # short TTL for "missing" sentinels (seconds)
POSITIVE_TTL = 300  # normal TTL for real data
NULL_SENTINEL = "__NULL__"

def naive_get_product(product_id, metrics):
    """Cache-aside WITHOUT negative caching — every miss hits the DB."""
    key = f"naive:product:{product_id}"
    cached = r.get(key)
    if cached:
        metrics.record("hit")
        return json.loads(cached)

    metrics.record("miss")
    metrics.record("db_query")
    conn = get_db(); cur = conn.cursor()
    cur.execute("SELECT id, name, price FROM products WHERE id = %s", (product_id,))
    row = cur.fetchone()
    conn.close()

    if row is None:
        return None  # ⚠️ we did NOT cache this — next call hits DB again
    product = {"id": row[0], "name": row[1], "price": float(row[2])}
    r.setex(key, POSITIVE_TTL, json.dumps(product))
    return product


def negative_cache_get_product(product_id, metrics):
    """Cache-aside WITH negative caching."""
    key = f"neg:product:{product_id}"
    cached = r.get(key)
    if cached == NULL_SENTINEL:
        metrics.record("negative_hit")
        return None
    if cached:
        metrics.record("hit")
        return json.loads(cached)

    metrics.record("miss")
    metrics.record("db_query")
    # Rule 3: never negative-cache an error/outage — let exceptions propagate.
    conn = get_db(); cur = conn.cursor()
    cur.execute("SELECT id, name, price FROM products WHERE id = %s", (product_id,))
    row = cur.fetchone()
    conn.close()

    if row is None:
        # Cache the "missing" fact with a short TTL
        r.setex(key, NEGATIVE_TTL, NULL_SENTINEL)
        return None

    product = {"id": row[0], "name": row[1], "price": float(row[2])}
    r.setex(key, POSITIVE_TTL, json.dumps(product))
    return product

print("✅ naive_get_product and negative_cache_get_product ready")

### Demo: 100 requests for a non-existent product

Imagine a bot scanning random IDs. Without negative caching, every request hits PostgreSQL. With it, only the first does.

In [ ]:
r.flushdb()
naive_metrics = CacheMetrics()
neg_metrics = CacheMetrics()

# Bad ID that doesn't exist
BAD_ID = 99999999

for _ in range(100):
    naive_get_product(BAD_ID, naive_metrics)

for _ in range(100):
    negative_cache_get_product(BAD_ID, neg_metrics)

print("WITHOUT negative caching:")
naive_metrics.report()
print()
print("WITH negative caching:")
neg_metrics.report()
print()
print("💡 Same 100 requests — without negative caching, 100 DB queries.")
print("   With it, just 1. That's the difference between a healthy DB and a 3am page.")

### What about real data appearing later?

The short `NEGATIVE_TTL` is your safety net. If product `99999999` *is* eventually created, the negative cache entry expires within `NEGATIVE_TTL` seconds (we used 30) and a fresh DB query picks up the new row. You trade a small staleness window for massive DB protection.

## ⏳ Pattern 2: Soft TTL vs Hard TTL (Stale-While-Revalidate)

**Problem:** With a normal TTL, the moment data expires the next request becomes a **cache miss** — slow, and (under load) prone to a stampede.

**Fix:** Give each cached entry **two timestamps**:
- **Soft expiry** (e.g. 5s): data is "fresh." Always served immediately.
- **Hard expiry** (e.g. 15s): data is "stale" but still usable. Serve it, then refresh.

If we're past the **hard expiry**, we block and refresh (just like a normal miss).

This is also called **stale-while-revalidate** (SWR) — the same idea HTTP caching uses.

We'll keep the demo **deterministic**: refreshes happen inline (not on a thread) so you can see exactly when they fire. In production you'd offload the refresh to a background worker so the requesting user never waits.

In [ ]:
SOFT_TTL = 5     # seconds: data is "fresh"
HARD_TTL = 15    # seconds: data is "stale but usable"

def fetch_product_from_db(product_id):
    conn = get_db(); cur = conn.cursor()
    cur.execute("SELECT id, name, price FROM products WHERE id = %s", (product_id,))
    row = cur.fetchone()
    conn.close()
    if row is None:
        return None
    return {"id": row[0], "name": row[1], "price": float(row[2])}


def _refresh(product_id, key, metrics):
    metrics.record("refresh")
    metrics.record("db_query")
    fresh = fetch_product_from_db(product_id)
    if fresh is None:
        return None
    payload = json.dumps({
        "data": fresh,
        "soft_expires_at": time.time() + SOFT_TTL,
    })
    r.setex(key, HARD_TTL, payload)
    return fresh


def swr_get_product(product_id, metrics):
    """
    Stale-While-Revalidate cache lookup.
    Cache value layout: {"data": <product>, "soft_expires_at": <epoch>}
    Redis TTL is set to HARD_TTL — Redis evicts the key past hard expiry.
    """
    key = f"swr:product:{product_id}"
    cached = r.get(key)

    if cached:
        entry = json.loads(cached)
        now = time.time()
        if now < entry["soft_expires_at"]:
            # Still fresh
            metrics.record("hit")
            return entry["data"]
        # Past soft expiry but Redis hasn't dropped it yet → stale-but-usable
        metrics.record("stale_serve")
        # In a real app the refresh would be async (e.g. enqueue a job).
        # We refresh inline here so the demo is deterministic.
        _refresh(product_id, key, metrics)
        return entry["data"]  # serve the stale copy NOW; refresh updated cache for next caller

    # Hard miss (Redis evicted the key past HARD_TTL)
    metrics.record("miss")
    return _refresh(product_id, key, metrics)

print("✅ swr_get_product ready (SOFT_TTL=5s, HARD_TTL=15s)")

### Demo: watch fresh / stale / refresh transitions

We'll read every 2 seconds and label each call.

In [ ]:
r.flushdb()
swr_metrics = CacheMetrics()

print("⏰ Stale-While-Revalidate timeline (SOFT=5s, HARD=15s):\n")
for t in range(0, 18, 2):
    if t > 0:
        time.sleep(2)
    before = dict(swr_metrics.counts)
    swr_get_product(42, swr_metrics)
    delta = {k: swr_metrics.counts[k] - before.get(k, 0)
             for k in swr_metrics.counts if swr_metrics.counts[k] - before.get(k, 0) > 0}
    label = ", ".join(f"{k}+{v}" for k, v in delta.items())
    print(f"   t={t:>2}s  →  {label}")

print()
swr_metrics.report()
print()
print("💡 First call: miss + refresh (cold).")
print("   Calls within SOFT_TTL: plain hits (instant).")
print("   Past SOFT_TTL but within HARD_TTL: stale_serve + background refresh.")
print("   Past HARD_TTL: Redis has evicted, full miss again.")

### Why this matters

A normal TTL forces *one unlucky request* to wait for the DB rebuild every expiry cycle. Under load, many requests pile up at the same instant — that's the stampede from Notebook 5.

With SWR:
- **Reads stay fast** — every caller gets a cached value (fresh or slightly stale).
- **Refresh cost is paid by one request** — and could even be done by a background worker.
- **No stampede** — others don't wait.

The trade-off: callers may see data up to `SOFT_TTL` seconds old. For most read paths (product pages, profiles, dashboards) this is fine.

## 🧰 Pattern 3: Read-Through-Style Wrapper

So far we've written cache-aside logic *inside* every read function. That gets tedious and error-prone — every new query repeats the same `check cache → query DB → set cache` dance.

A **read-through-style wrapper** hides that dance behind a clean API:

```python
products = ProductRepo(cache, metrics)
product = products.get(42)   # cache logic happens inside
```

> ⚠️ **Note on terminology**: A *true* read-through cache is one where the cache layer itself loads from the DB on miss (e.g. some commercial caching products). Redis doesn't do that. What we're building is an **application-level wrapper** that gives the *same ergonomic feel* on top of cache-aside. Beginners often confuse the two — now you won't.

In [ ]:
class ProductRepo:
    """
    Read-through-style wrapper around cache-aside.
    The caller doesn't see the cache at all — just .get() and .invalidate().
    """

    TTL = 300

    def __init__(self, cache, metrics):
        self.cache = cache
        self.metrics = metrics

    def _key(self, product_id):
        return f"repo:product:{product_id}"

    def get(self, product_id):
        key = self._key(product_id)
        cached = self.cache.get(key)
        if cached:
            self.metrics.record("hit")
            return json.loads(cached)

        self.metrics.record("miss")
        self.metrics.record("db_query")
        product = fetch_product_from_db(product_id)
        if product is None:
            return None
        self.cache.setex(key, self.TTL, json.dumps(product))
        return product

    def invalidate(self, product_id):
        self.cache.delete(self._key(product_id))


# Demo: caller code is dramatically simpler
r.flushdb()
repo_metrics = CacheMetrics()
repo = ProductRepo(r, repo_metrics)

print("📦 First read (cold):")
print(f"   {repo.get(42)}")
print("\n📦 Second read (warm):")
print(f"   {repo.get(42)}")
print("\n📦 After invalidate:")
repo.invalidate(42)
print(f"   {repo.get(42)}")
print()
repo_metrics.report()
print()
print("💡 Notice: nowhere in the demo did we touch Redis directly.")
print("   This is the value of a wrapper — one place to change cache policy.")

### Comparison: cache-aside vs application-level read-through wrapper

| Aspect | Plain cache-aside | Wrapper (this notebook) | True read-through (e.g. EHCache) |
|---|---|---|---|
| Who calls the cache? | App code, every time | Wrapper | Cache library itself |
| Who calls the DB on miss? | App code | Wrapper | Cache library |
| Repeat boilerplate? | Yes | No | No |
| Works with Redis? | ✅ | ✅ | ❌ (Redis doesn't load from your DB) |

For real systems, the wrapper pattern is what most teams actually ship.

## 🔥 Pattern 4: Cache Warming

**Problem:** When you start a fresh app server (or cache), the cache is empty. The first wave of users all see cache misses → the DB takes a hit just when traffic is recovering.

**Fix:** **Pre-populate** the cache at startup with the data you know will be hot. For an e-commerce catalog, that's usually the **top-N most-viewed products**.

### Rules
- Keep `N` **small and bounded** — you're not trying to cache everything.
- Make warming **idempotent** — running it twice should be safe.
- Treat it as a **trade-off**: slightly slower startup, much smoother first minute.

In [ ]:
def warm_top_products(repo, n=20):
    """Pre-fetch the N most-viewed products into the cache."""
    conn = get_db(); cur = conn.cursor()
    cur.execute(
        "SELECT id FROM products ORDER BY view_count DESC LIMIT %s",
        (n,)
    )
    ids = [row[0] for row in cur.fetchall()]
    conn.close()

    for pid in ids:
        repo.get(pid)  # populates the cache via the wrapper

    return len(ids)


# Demo: cold cache vs warmed cache
r.flushdb()
cold_metrics = CacheMetrics()
warm_metrics = CacheMetrics()

# --- Pick the same set of "hot" products both runs will receive
conn = get_db(); cur = conn.cursor()
cur.execute("SELECT id FROM products ORDER BY view_count DESC LIMIT 20")
top_ids = [row[0] for row in cur.fetchall()]
conn.close()

# --- Cold cache: simulate user traffic on top products
cold_repo = ProductRepo(r, cold_metrics)
t0 = time.time()
for _ in range(100):
    cold_repo.get(random.choice(top_ids))
cold_time = (time.time() - t0) * 1000

# --- Warmed cache: warm first, then same traffic
r.flushdb()
warm_repo = ProductRepo(r, warm_metrics)
warm_count = warm_top_products(warm_repo, n=20)
# Reset metrics so we only measure user traffic, not warming
warm_metrics.counts.clear()

t0 = time.time()
for _ in range(100):
    warm_repo.get(random.choice(top_ids))
warm_time = (time.time() - t0) * 1000

print(f"❄️  COLD start  → 100 requests took {cold_time:.1f}ms")
cold_metrics.report()
print()
print(f"🔥 WARMED ({warm_count} pre-loaded) → 100 requests took {warm_time:.1f}ms")
warm_metrics.report()
print()
print(f"💡 Warming traded ~{warm_count} startup queries for a 100% hit rate on hot reads.")

### When NOT to warm

- **Long tail data** — caching things nobody asks for is just memory waste.
- **Per-user data** — you can't pre-warm 10M user profiles.
- **Frequently-changing data** — you'd just be caching stale data sooner.

Warming is for **small, hot, mostly-read** data: top products, configuration, feature flags, popular search results.

## 🧹 Cleanup

In [ ]:
r.flushdb()
print("🧹 Redis cleared")

## 📚 Summary

| Pattern | Problem it solves | Key trade-off |
|---|---|---|
| **Negative caching** | Bots/typos hammer DB for missing keys | Briefly serves "missing" if item is created later |
| **Soft / Hard TTL (SWR)** | TTL expiry causes slow reads + stampedes | Callers may see data up to `SOFT_TTL` old |
| **Read-through wrapper** | Cache logic duplicated everywhere | Slight abstraction cost; not "true" read-through |
| **Cache warming** | Cold caches punish the DB at startup | Slower startup, bounded cost |

### Key takeaways

1. **Always cache "not found"** — use a sentinel + short TTL. Cache penetration is a real attack vector.
2. **Stale data is often better than slow data** — soft/hard TTL keeps reads fast and absorbs refresh cost.
3. **Hide cache logic behind a wrapper** once you have more than one or two cached entities.
4. **Warm the things that matter** — small set, idempotent, bounded.
5. **Measure everything** — without metrics, you can't tell whether your cache is helping.

### You've completed the caching lab! 🎉

You now understand: why caching matters, cache-aside, write-through/behind, invalidation, TTL, eviction, stampedes, hot keys, negative caching, stale-while-revalidate, read-through wrappers, and cache warming.

Next steps:
- Re-read your favorite system design problem and identify *where* caching would help and *which pattern* fits.
- Try replacing the in-process metrics dict with [Prometheus](https://prometheus.io) counters.
- Explore Redis features we didn't cover: pub/sub, streams, sorted sets for leaderboards.
